In [1]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

# treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own"]
)

connected to port: 50855


Find the already annotated polycategorial lexical units (such as divers det,adj or from adp, sconj)

In [2]:
previous_lemma_pos = list(corpus._idx2lexunit.values())[0]
polycategorial_lexunits = {}
for lemma, pos in list(corpus._idx2lexunit.values())[1:]:
    current_lemma_pos = (lemma, pos)
    if current_lemma_pos[0] == previous_lemma_pos[0]:
        if current_lemma_pos[0] not in polycategorial_lexunits:
            polycategorial_lexunits[current_lemma_pos[0]] = (current_lemma_pos[1], previous_lemma_pos[1])
        else:
            polycategorial_lexunits[current_lemma_pos[0]] += (current_lemma_pos[1],)
    previous_lemma_pos = current_lemma_pos
print(polycategorial_lexunits)
print(len(polycategorial_lexunits))
    
    

{'American': ('PROPN', 'ADJ'), 'English': ('PROPN', 'ADJ'), 'French': ('PROPN', 'ADJ'), 'New': ('PROPN', 'ADJ'), 'about': ('ADV', 'ADP', 'SCONJ'), 'after': ('SCONJ', 'ADP'), 'all': ('DET', 'ADV'), 'along': ('ADV', 'ADP'), 'around': ('ADV', 'ADP'), 'as': ('ADV', 'ADP', 'SCONJ'), 'at': ('SCONJ', 'ADP'), 'attack': ('VERB', 'NOUN'), 'attempt': ('VERB', 'NOUN'), 'back': ('NOUN', 'ADV'), 'base': ('VERB', 'NOUN'), 'be': ('VERB', 'AUX'), 'bear': ('VERB', 'NOUN'), 'before': ('ADV', 'ADP', 'SCONJ'), 'benefit': ('VERB', 'NOUN'), 'both': ('DET', 'CCONJ'), 'break': ('VERB', 'NOUN'), 'by': ('SCONJ', 'ADP'), 'call': ('VERB', 'NOUN'), 'can': ('NOUN', 'AUX'), 'cause': ('SCONJ', 'NOUN', 'VERB'), 'challenge': ('VERB', 'NOUN'), 'change': ('VERB', 'NOUN'), 'check': ('VERB', 'NOUN'), 'chemical': ('NOUN', 'ADJ'), 'claim': ('VERB', 'NOUN'), 'clean': ('VERB', 'ADJ'), 'close': ('ADV', 'ADJ', 'VERB'), 'complete': ('VERB', 'ADJ'), 'contact': ('VERB', 'NOUN'), 'deal': ('VERB', 'NOUN'), 'design': ('VERB', 'NOUN'), 

In [3]:
pos_nbelements = {}
for lemma, pos in list(corpus._idx2lexunit.values())[0:]:
    if pos not in pos_nbelements:
        pos_nbelements[pos] = {"total": 0, "polycategorial": 0}
    pos_nbelements[pos]["total"] += 1
    if lemma in polycategorial_lexunits:
        pos_nbelements[pos]["polycategorial"] += 1
print(pos_nbelements)

{'SYM': {'total': 4, 'polycategorial': 0}, 'CCONJ': {'total': 8, 'polycategorial': 3}, 'PART': {'total': 3, 'polycategorial': 1}, 'ADP': {'total': 52, 'polycategorial': 27}, 'NUM': {'total': 70, 'polycategorial': 4}, 'NOUN': {'total': 856, 'polycategorial': 98}, 'PROPN': {'total': 214, 'polycategorial': 4}, 'ADJ': {'total': 286, 'polycategorial': 48}, 'PRON': {'total': 38, 'polycategorial': 6}, 'VERB': {'total': 375, 'polycategorial': 85}, 'DET': {'total': 17, 'polycategorial': 10}, 'ADV': {'total': 141, 'polycategorial': 44}, 'SCONJ': {'total': 29, 'polycategorial': 25}, 'INTJ': {'total': 30, 'polycategorial': 7}, 'X': {'total': 2, 'polycategorial': 0}, 'AUX': {'total': 13, 'polycategorial': 6}}


In [61]:
corpus.lexunit2idx(('soit', 'ADV'))

2506

Measure the distance between the vectors representing the two lexical units (e.g. v1 = anglais, noun and v2 = anglais, adj) and compare the cross-POS distance to the average distance within the category. A split is justified if the distance between (anglais, N) and (anglais, adj) is significantly higher than the typical distance between two nouns.

In [4]:
def _get_category_stats(target_pos, target_idx, target_vec, metric='cosine'):
        # Indices of all other members of this POS
        cat_indices = [idx for idx, unit in idx_to_unit.items() 
                       if unit[1] == target_pos and idx != target_idx]
        
        if len(cat_indices) < 2:
            return 0.0, 0.0, 0.0 # Not enough data to compute stats
        
        # Distances from our target word to all other words in its POS
        cat_matrix = matrix[cat_indices]
        dists = cdist(target_vec, cat_matrix, metric=metric)[0]
        
        return np.mean(dists), np.std(dists), len(cat_indices)

In [5]:
import numpy as np
from scipy.spatial.distance import cdist

def check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, lemma, poses, metric='cosine', k=1.0):
    """
    Checks if dist(lemma_pos1, lemma_pos2) is greater than the intra-category 
    distance for BOTH categories.
    
    Args:
        matrix (np.array): N x M matrix (rows=units, cols=features).
        idx_to_unit (dict): Maps row index -> (lemma, pos).
        unit_to_idx (dict): Maps (lemma, pos) -> row index.
        lemma (str): The lemma to check.
        poses (tuple): A tuple of two POS tags (pos1, pos2).
        metric (str): Distance metric ('cosine', 'euclidean', etc.).
        
    Returns:
        dict: Results containing distances, the comparison boolean, and the farthest element.
    """
    
    target_1 = (lemma, poses[0])
    target_2 = (lemma, poses[1])

    idx1 = unit_to_idx[target_1]
    idx2 = unit_to_idx[target_2]

    vec1 = matrix[idx1].reshape(1, -1) # Reshape for cdist
    vec2 = matrix[idx2].reshape(1, -1)

    # 3. Calculate Cross-POS distance (e.g., dist(dance_N, dance_V))
    # cdist returns a 2D array, we take [0][0]
    cross_dist = cdist(vec1, vec2, metric=metric)[0][0]

    mu1, std1, count1 = _get_category_stats(poses[0], idx1, vec1, metric)
    mu2, std2, count2 = _get_category_stats(poses[1], idx2, vec2, metric)

    # 4. Calculate Z-scores
    # How many standard deviations is the cross_dist away from the category mean?
    z1 = (cross_dist - mu1) / std1 if std1 > 0 else 0
    z2 = (cross_dist - mu2) / std2 if std2 > 0 else 0

    # 5. Final Logic
    # We split if the cross_dist is significantly larger than the average variation in BOTH categories.
    # should_split = (z1 > k) or (z2 > k)
    should_split = cross_dist > mu1 or cross_dist > mu2

    return {
        "lemma": lemma,
        "pos1": poses[0],
        "pos2": poses[1],
        "cross_dist": round(float(cross_dist), 4),
        f"{poses[0]}_stats": {"mean": round(float(mu1), 4), "std": round(float(std1), 4), "z_score": round(float(z1), 2)},
        f"{poses[1]}_stats": {"mean": round(float(mu2), 4), "std": round(float(std2), 4), "z_score": round(float(z2), 2)},
        "should_split": bool(should_split),
        "threshold_k": k
    }

# --- Example Usage ---

# # 1. Create Dummy Data
# # 5 lexical units, 3 features each
matrix = np.array([
    [1.0, 0.9, 0.1],  # 0: (dance, noun)
    [0.1, 0.2, 0.9],  # 1: (dance, verb) - Very different from noun
    [0.9, 0.8, 0.2],  # 2: (table, noun) - Similar to dance_noun
    [0.8, 0.9, 0.1],  # 3: (chair, noun) - Similar to dance_noun
    [0.5, 0.5, 0.5],  # 4: (weird, noun) - The "farthest" noun
])

idx_to_unit = {
    0: ("dance", "noun"),
    1: ("dance", "verb"),
    2: ("table", "noun"),
    3: ("chair", "noun"),
    4: ("weird", "noun")
}

unit_to_idx = {v: k for k, v in idx_to_unit.items()}
# 2. Run the check
result = check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, "dance", ("noun","verb"), metric='cosine', k=1.0)

# 3. Print Results
print(result)
# print(f"Distance (dance, N) <-> (dance, V): {result['cross_dist']:.4f}")
# print(f"Max Dist (dance, N) <-> Any Noun:   {result['max_intra_cat_dist']:.4f}")
# print(f"Farthest Noun:                      {result['farthest_same_cat_unit']}")
# print("-" * 30)
# print(f"Is Cross Distance Bigger?           {result['is_cross_bigger']}")

{'lemma': 'dance', 'pos1': 'noun', 'pos2': 'verb', 'cross_dist': 0.7043, 'noun_stats': {'mean': 0.0515, 'std': 0.0655, 'z_score': 9.97}, 'verb_stats': {'mean': 0.0, 'std': 0.0, 'z_score': 0.0}, 'should_split': True, 'threshold_k': 1.0}


In [70]:
True or False

True

In [6]:
matrix = corpus.feature_matrix
idx_to_unit = corpus._idx2lexunit
unit_to_idx = corpus._lexunit2idx
for lemma, poses in polycategorial_lexunits.items():
    comparison_result = check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, lemma, poses, metric='cosine', k=1.0)
    # print(f"{comparison_result['lemma']}, {comparison_result['pos1']} vs {comparison_result['pos2']}: should split? {comparison_result['should_split']}")
    print(comparison_result)

    # print(f"{dico['lemma']}, {dico['pos1']} vs {dico['pos2']}: {decision}")

{'lemma': 'American', 'pos1': 'PROPN', 'pos2': 'ADJ', 'cross_dist': 0.5315, 'PROPN_stats': {'mean': 0.3814, 'std': 0.1362, 'z_score': 1.1}, 'ADJ_stats': {'mean': 0.2558, 'std': 0.2401, 'z_score': 1.15}, 'should_split': True, 'threshold_k': 1.0}
{'lemma': 'English', 'pos1': 'PROPN', 'pos2': 'ADJ', 'cross_dist': 0.5755, 'PROPN_stats': {'mean': 0.3887, 'std': 0.1501, 'z_score': 1.24}, 'ADJ_stats': {'mean': 0.2369, 'std': 0.2246, 'z_score': 1.51}, 'should_split': True, 'threshold_k': 1.0}
{'lemma': 'French', 'pos1': 'PROPN', 'pos2': 'ADJ', 'cross_dist': 0.4395, 'PROPN_stats': {'mean': 0.4019, 'std': 0.1217, 'z_score': 0.31}, 'ADJ_stats': {'mean': 0.2654, 'std': 0.2031, 'z_score': 0.86}, 'should_split': True, 'threshold_k': 1.0}
{'lemma': 'New', 'pos1': 'PROPN', 'pos2': 'ADJ', 'cross_dist': 0.2919, 'PROPN_stats': {'mean': 0.4449, 'std': 0.1458, 'z_score': -1.05}, 'ADJ_stats': {'mean': 0.5356, 'std': 0.1795, 'z_score': -1.36}, 'should_split': False, 'threshold_k': 1.0}
{'lemma': 'about', 'po

In [7]:
import numpy as np

def get_discriminating_features(matrix, unit_to_idx, idx_to_feature, lemma, poses, top_n=10):
    """
    Identifies which features (columns) have the largest difference between two senses.
    """
    pos1, pos2 = poses
    target_1, target_2 = (lemma, pos1), (lemma, pos2)

    # 1. Extract the vectors
    idx1 = unit_to_idx[target_1]
    idx2 = unit_to_idx[target_2]
    
    vec1 = matrix[idx1]
    vec2 = matrix[idx2]

    # 2. Calculate absolute difference per feature: |v1 - v2|
    # Formula: $\Delta f = |v_{1,f} - v_{2,f}|$
    diffs = np.abs(vec1 - vec2)

    # 3. Get indices of the largest differences
    # argsort sorts ascending, so we take the end of the array and reverse it
    top_indices = np.argsort(diffs)[-top_n:][::-1]

    results = []
    for i in top_indices:
        feature_name = idx_to_feature[i]
        results.append({
            "feature": feature_name,
            f"val_{pos1}": round(float(vec1[i]), 4),
            f"val_{pos2}": round(float(vec2[i]), 4),
            "abs_diff": round(float(diffs[i]), 4)
        })

    return results

In [8]:
top_diffs = get_discriminating_features(matrix, unit_to_idx, corpus._idx2feature, "of", ("SCONJ", "ADP"))

for d in top_diffs:
    print(d)

{'feature': 'node:X:parent:upos=VERB', 'val_SCONJ': 0.1249, 'val_ADP': 0.0004, 'abs_diff': 0.1245}
{'feature': 'node:X:parent:VerbForm=Ger', 'val_SCONJ': 0.1058, 'val_ADP': 0.0, 'abs_diff': 0.1058}
{'feature': 'node:X:next:upos=VERB', 'val_SCONJ': 0.1048, 'val_ADP': 0.0023, 'abs_diff': 0.1025}
{'feature': 'node:X:next:VerbForm=Ger', 'val_SCONJ': 0.0952, 'val_ADP': 0.0004, 'abs_diff': 0.0949}
{'feature': 'node:X:parent:upos=NOUN', 'val_SCONJ': 0.0042, 'val_ADP': 0.0888, 'abs_diff': 0.0846}
{'feature': 'node:X:parent:Number=Sing', 'val_SCONJ': 0.0063, 'val_ADP': 0.0801, 'abs_diff': 0.0737}
{'feature': 'node:X:next:Number=Sing', 'val_SCONJ': 0.0032, 'val_ADP': 0.0504, 'abs_diff': 0.0472}
{'feature': 'node:X:next:upos=DET', 'val_SCONJ': 0.0021, 'val_ADP': 0.0391, 'abs_diff': 0.037}
{'feature': 'node:X:parent:Number=Plur', 'val_SCONJ': 0.0011, 'val_ADP': 0.0358, 'abs_diff': 0.0348}
{'feature': 'node:X:next:upos=NOUN', 'val_SCONJ': 0.0021, 'val_ADP': 0.0347, 'abs_diff': 0.0325}


In [9]:
from itertools import combinations
import numpy as np
from scipy.spatial.distance import cdist

def check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine'):
    # Generate all unique pairs of poses
    # e.g., if poses=['NOUN', 'VERB', 'ADJ'], pairs are (N, V), (N, A), (V, A)
    pose_pairs = list(combinations(poses, 2))
    results = []

    def get_local_density(target_pos, target_idx, target_vec):
        all_cat_indices = [idx for idx, u in idx_to_unit.items() if u[1] == target_pos and idx != target_idx]
        
        if not all_cat_indices:
            return 0.0
            
        if len(all_cat_indices) < k_neighbors:
            compare_indices = all_cat_indices
        else:
            all_dists = cdist(target_vec, matrix[all_cat_indices], metric=metric)[0]
            closest_k_relative_indices = np.argsort(all_dists)[:k_neighbors]
            compare_indices = [all_cat_indices[i] for i in closest_k_relative_indices]
        
        local_dists = cdist(target_vec, matrix[compare_indices], metric=metric)[0]
        return np.median(local_dists)

    # Iterate through each pair and calculate the split strength
    for pos1, pos2 in pose_pairs:
        t1, t2 = (lemma, pos1), (lemma, pos2)
        
        # Skip this pair if one of the senses isn't in our mapping
        if t1 not in unit_to_idx or t2 not in unit_to_idx:
            continue

        v1 = matrix[unit_to_idx[t1]].reshape(1, -1)
        v2 = matrix[unit_to_idx[t2]].reshape(1, -1)
        cross_dist = cdist(v1, v2, metric=metric)[0][0]

        local_size1 = get_local_density(pos1, unit_to_idx[t1], v1)
        local_size2 = get_local_density(pos2, unit_to_idx[t2], v2)

        robust_ratio1 = cross_dist / local_size1 if local_size1 > 0 else 0
        robust_ratio2 = cross_dist / local_size2 if local_size2 > 0 else 0
        split_strength = min(robust_ratio1, robust_ratio2)

        min_ratio = min(robust_ratio1, robust_ratio2)
        difference = abs(robust_ratio1 - robust_ratio2)
        split_quality_score = min_ratio / (1 + difference )

        results.append({
            "lemma": lemma,
            "poses_compared": (pos1, pos2), # Added for clarity in multi-pose output
            "split_strength": round(split_strength, 3),
            "interpretation": "High = Clear Split | Low = Merged/Noisy",
            "ratios": {pos1: round(robust_ratio1, 2), pos2: round(robust_ratio2, 2)},
            "local_densities": {pos1: round(local_size1, 3), pos2: round(local_size2, 3)},
            "cross_dist": round(cross_dist, 3),
            "min_ratio": round(min_ratio, 3),
            "difference" : round(difference, 3),
            "split_quality_score": round(split_quality_score, 3)
        })

    # Return the list of results (or None if no valid pairs were found)
    return results if results else None

In [10]:
for lemma, poses in polycategorial_lexunits.items():
    result = check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine')
    for dico in result:
        print(dico)

{'lemma': 'American', 'poses_compared': ('PROPN', 'ADJ'), 'split_strength': 2.868, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'PROPN': 2.87, 'ADJ': 10.03}, 'local_densities': {'PROPN': 0.185, 'ADJ': 0.053}, 'cross_dist': 0.532, 'min_ratio': 2.868, 'difference': 7.163, 'split_quality_score': 0.351}
{'lemma': 'English', 'poses_compared': ('PROPN', 'ADJ'), 'split_strength': 5.833, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'PROPN': 5.83, 'ADJ': 13.91}, 'local_densities': {'PROPN': 0.099, 'ADJ': 0.041}, 'cross_dist': 0.575, 'min_ratio': 5.833, 'difference': 8.076, 'split_quality_score': 0.643}
{'lemma': 'French', 'poses_compared': ('PROPN', 'ADJ'), 'split_strength': 2.272, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'PROPN': 2.27, 'ADJ': 6.38}, 'local_densities': {'PROPN': 0.193, 'ADJ': 0.069}, 'cross_dist': 0.44, 'min_ratio': 2.272, 'difference': 4.104, 'split_quality_score': 0.445}
{'lemma': 'New', 'pos

In [11]:
results = []
for lemma, poses in polycategorial_lexunits.items():
    result = check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine')
    for dico in result:
        results.extend(result)

In [12]:
results

[{'lemma': 'American',
  'poses_compared': ('PROPN', 'ADJ'),
  'split_strength': 2.868,
  'interpretation': 'High = Clear Split | Low = Merged/Noisy',
  'ratios': {'PROPN': 2.87, 'ADJ': 10.03},
  'local_densities': {'PROPN': 0.185, 'ADJ': 0.053},
  'cross_dist': 0.532,
  'min_ratio': 2.868,
  'difference': 7.163,
  'split_quality_score': 0.351},
 {'lemma': 'English',
  'poses_compared': ('PROPN', 'ADJ'),
  'split_strength': 5.833,
  'interpretation': 'High = Clear Split | Low = Merged/Noisy',
  'ratios': {'PROPN': 5.83, 'ADJ': 13.91},
  'local_densities': {'PROPN': 0.099, 'ADJ': 0.041},
  'cross_dist': 0.575,
  'min_ratio': 5.833,
  'difference': 8.076,
  'split_quality_score': 0.643},
 {'lemma': 'French',
  'poses_compared': ('PROPN', 'ADJ'),
  'split_strength': 2.272,
  'interpretation': 'High = Clear Split | Low = Merged/Noisy',
  'ratios': {'PROPN': 2.27, 'ADJ': 6.38},
  'local_densities': {'PROPN': 0.193, 'ADJ': 0.069},
  'cross_dist': 0.44,
  'min_ratio': 2.272,
  'difference': 4

In [13]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)

# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))

# Create the plot
fig = px.scatter(
    df, 
    x="cross_dist", 
    y="split_strength",
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities"
    },
    title="Lexical Split Analysis: Cross Distance vs Split Strength"
)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

/opt/homebrew/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [14]:
fig.write_html("en_gsd_splitstrength_vs_crossdistance.html")

In [15]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)

# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))

# Create the plot
fig = px.scatter(
    df, 
    x="split_strength", 
    y="cross_dist",
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "split_quality_score": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities",
        "split_quality_score": "Min ratio/difference"
    },
    title="Lexical Split Analysis: Split strength vs cross_dist (ss = min(r1, r2) where r_x = cross_dist / median dist between x and 20 closest neighbours)"
)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [16]:
fig.write_html("en_cross_dist_vs_split_strength.html")

In [17]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)

# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))

# Create the plot
fig = px.scatter(
    df, 
    x="split_strength", 
    y="cross_dist",
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "split_quality_score": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities",
        "split_quality_score": "Min ratio/difference"
    },
    title="Lexical Split Analysis: Split strength vs cross_dist (ss = min(r1, r2) where r_x = cross_dist / median dist between x and 20 closest neighbours)"
)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [18]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)

# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
x_series = df['ratios'].apply(lambda x: list(x.values())[0])
y_series = df['ratios'].apply(lambda x: list(x.values())[1])
# Create the plot
fig = px.scatter(
    df, 
    x= df['ratios'].apply(lambda x: list(x.values())[0]), 
    y=df['ratios'].apply(lambda x: list(x.values())[1]), 
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "split_quality_score": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities",
        "split_quality_score": "Min ratio/difference"
    },
    title="Lexical Split Analysis: Split quality score (min ratio/difference) vs. Split strength "
)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)
min_val = float(min(x_series.min(), y_series.min()))
max_val = float(max(x_series.max(), y_series.max()))
fig.add_shape(type="line", x0=min_val, y0=min_val, x1=max_val, y1=max_val, xref="x", yref="y", line=dict(color="gray", dash="dash"))

fig.show()

In [19]:
def get_split_quality(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20):
    # ... (Use the existing logic to get ratios) ...
    res = check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors)
    
    r1 = res['ratios'][poses[0]]
    r2 = res['ratios'][poses[1]]
    
    # 1. Separation (How far are they?)
    separation = min(r1, r2)
    
    # 2. Asymmetry (How unbalanced is the split?)
    asymmetry = abs(r1 - r2)
    
    # 3. Final Quality Score
    # We penalize high asymmetry
    quality_score = separation / (1 + asymmetry)
    
    return {
        "lemma": lemma,
        "split_quality": round(quality_score, 3),
        "separation": round(separation, 3),
        "asymmetry": round(asymmetry, 3),
        "verdict": "Clean Split" if quality_score > 1.5 else "Messy/Asymmetric" if quality_score > 0.5 else "Merge"
    }

In [20]:
for lemma, poses in polycategorial_lexunits.items():
    result = check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine')
    for dico in result:
        current_poses = list(dico['ratios'].keys())
        r1 = dico['ratios'][current_poses[0]]
        r2 = dico['ratios'][current_poses[1]]

        min_ratio = min(r1, r2)
        difference = abs(r1 - r2)
        split_quality_score = min_ratio / (1 + difference ) # logic: if min ratio is high and both ratios are kind of similar then the two functions are far apart, so split. if the difference is high, so the min ratio is a lot smaller than the other one then the point
        # clearly belongs to one of the category more than the other, so the split is poor. if the ratios are low but they're both low it's a grey area, and there should probably be a third category there. 

        split_quality_output = {
        "lemma": lemma,
        "split_quality": round(split_quality_score, 3),
        "separation": round(min_ratio, 3),
        "asymmetry": round(difference, 3),
        "verdict": "Clean Split" if split_quality_score > 1.5 else "Messy/Asymmetric" if split_quality_score > 0.5 else "Merge"
    }
        print(split_quality_output)

{'lemma': 'American', 'split_quality': 0.352, 'separation': 2.87, 'asymmetry': 7.16, 'verdict': 'Merge'}
{'lemma': 'English', 'split_quality': 0.642, 'separation': 5.83, 'asymmetry': 8.08, 'verdict': 'Messy/Asymmetric'}
{'lemma': 'French', 'split_quality': 0.444, 'separation': 2.27, 'asymmetry': 4.11, 'verdict': 'Merge'}
{'lemma': 'New', 'split_quality': 0.924, 'separation': 1.33, 'asymmetry': 0.44, 'verdict': 'Messy/Asymmetric'}
{'lemma': 'about', 'split_quality': 0.138, 'separation': 1.04, 'asymmetry': 6.54, 'verdict': 'Merge'}
{'lemma': 'about', 'split_quality': 0.373, 'separation': 1.28, 'asymmetry': 2.43, 'verdict': 'Merge'}
{'lemma': 'about', 'split_quality': 0.538, 'separation': 3.0, 'asymmetry': 4.58, 'verdict': 'Messy/Asymmetric'}
{'lemma': 'after', 'split_quality': 0.406, 'separation': 3.49, 'asymmetry': 7.6, 'verdict': 'Merge'}
{'lemma': 'all', 'split_quality': 0.784, 'separation': 1.2, 'asymmetry': 0.53, 'verdict': 'Messy/Asymmetric'}
{'lemma': 'along', 'split_quality': 0.2

In [21]:
import pandas as pd
import plotly.express as px

# 1. Define your thresholds
# 1.0 is the theoretical threshold for Split Strength
ss_thresh = df['split_strength'].median()
# We use the median for Cross Distance to make it relative to your data distribution
cd_thresh = df['cross_dist'].median() 

# 2. Categorize the lemmas into the four quadrants
def categorize(row):
    if row['split_strength'] >= ss_thresh:
        return "Robust Split" if row['cross_dist'] >= cd_thresh else "Mini-Cluster"
    else:
        return "Sparse/Noisy" if row['cross_dist'] >= cd_thresh else "Merged/Syncretic"

df['quadrant'] = df.apply(categorize, axis=1)

# 3. Create the plot
fig = px.scatter(
    df, 
    x="split_strength", 
    y="cross_dist",
    color="quadrant", # This automatically handles the colors
    text="lemma",
    hover_name="lemma",
    color_discrete_map={
        "Robust Split": "#2ca02c",    # Green
        "Mini-Cluster": "#1f77b4",    # Blue
        "Sparse/Noisy": "#ff0efb",    # Orange
        "Merged/Syncretic": "#d62728" # Red
    },
    hover_data={
        "cross_dist": ":.3f",
        "split_strength": ":.3f",
        "poses_str": True,
        "split_quality_score": ":.3f",
        "quadrant": True,
        "lemma": False 
    },
    labels={
        "cross_dist": "Cross Distance (CD)",
        "split_strength": "Split Strength (SS)",
        "quadrant": "Interpretation"
    },
    title="Lexical Split Analysis: Quadrant Mapping"
)

# 4. Add the quadrant divider lines
fig.add_vline(x=ss_thresh, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=cd_thresh, line_dash="dash", line_color="gray", opacity=0.5)

# 5. Add explanatory labels in the corners
fig.add_annotation(x=df['split_strength'].max(), y=df['cross_dist'].max(),
                text="<b>TRUE HETEROSEMY</b><br>Far apart & well-anchored", showarrow=False, font=dict(color="green"))
fig.add_annotation(x=0, y=df['cross_dist'].max(),
                text="<b>SPARSE OUTLIERS</b><br>Far but categories are loose", showarrow=False, font=dict(color="orange"), xanchor="left")
fig.add_annotation(x=df['split_strength'].max(), y=df['cross_dist'].min(),
                text="<b>SUBTLE DISTINCTION</b><br>Close but very tight clusters", showarrow=False, font=dict(color="blue"))
fig.add_annotation(x=0, y=df['cross_dist'].min(),
                text="<b>MERGED</b><br>No functional difference", showarrow=False, font=dict(color="red"), xanchor="left")

# Final Styling
fig.update_traces(textposition='top center')
fig.update_layout(
    height=800,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

In [22]:
fig.write_html("en_cross_dist_vs_split_strength_annotated.html")

In [23]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)
ss_thresh = df['split_strength'].median()
# We use the median for Cross Distance to make it relative to your data distribution
d_thresh = df['difference'].median() 
# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))

# Create the plot
fig = px.scatter(
    df, 
    x="difference", 
    y="split_strength",
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "split_quality_score": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities",
        "split_quality_score": "Min ratio/difference"
    },
    title="Lexical Split Analysis: Split strength vs Asymmetry"
)

fig.add_vline(x=ss_thresh, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=d_thresh, line_dash="dash", line_color="gray", opacity=0.5)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [24]:
fig.write_html("en_split_strength_vs_difference.html")

In [25]:
import pandas as pd
import plotly.express as px


# Convert to DataFrame
df = pd.DataFrame(results)
sq_thresh = df['split_quality_score'].median()
# We use the median for Cross Distance to make it relative to your data distribution
cd_thresh = df['cross_dist'].median() 
# Create a readable string for the hover info
df['poses_str'] = df['poses_compared'].apply(lambda x: f"{x[0]} vs {x[1]}")
df['ratios_str'] = df['ratios'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))
df['densities_str'] = df['local_densities'].apply(lambda x: ", ".join([f"{k}: {v}" for k, v in x.items()]))

# Create the plot
fig = px.scatter(
    df, 
    x="cross_dist", 
    y="split_quality_score",
    text="lemma",  # This annotates the points
    hover_name="lemma",
    hover_data={
        "cross_dist": True,
        "split_strength": True,
        "poses_str": True,
        "ratios_str": True,
        "densities_str": True,
        "split_quality_score": True,
        "lemma": False # Hide it from the list since it's the title
    },
    labels={
        "cross_dist": "Cross Distance Score",
        "split_strength": "Split Strength Score",
        "poses_str": "POS Compared",
        "ratios_str": "Ratios",
        "densities_str": "Local Densities",
        "split_quality_score": "Split Quality Score"
    },
    title="Lexical Split Analysis: Split Quality vs Cross Distance"
)

fig.add_vline(x=sq_thresh, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=cd_thresh, line_dash="dash", line_color="gray", opacity=0.5)

# Improve text positioning and appearance
fig.update_traces(textposition='top center')
fig.update_layout(
    height=700,
    template="plotly_white",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [26]:
fig.write_html("en_splitquality_vs_crossdistance.html")

In [27]:
# 2. Create the POS Pair label for the X-axis
# This turns ('NOUN', 'ADJ') into "NOUN - ADJ"
df = pd.DataFrame(results)
df['pos_pair'] = df['poses_compared'].apply(lambda x: " - ".join(sorted(list(x))))
sq_thresh = df['split_quality_score'].median()
# 3. Generate the Category Health Box Plot
fig = px.box(
    df, 
    x="pos_pair", 
    y="split_quality_score", 
    points="all",           # Shows individual lemma dots
    hover_name="lemma",     # Lets you see which word is which dot
    color="pos_pair",
    title="Category Health Check: Distribution of Split Quality by POS Pair",
    labels={"pos_pair": "POS Relationship", "split_quality_score": "Split Quality"}
)

# Optional: Add a reference line at Quality = 1.0 (The "Ambiguity Line")
fig.add_hline(y=sq_thresh, line_dash="dot", line_color="red", annotation_text="Quality Median Value")
ordered_indices = df.groupby("pos_pair")["split_quality_score"].median().sort_values().index
fig.update_xaxes(categoryorder='array', categoryarray=ordered_indices)
fig.show()

In [28]:
import pandas as pd
import plotly.express as px

# 1. Prepare and Deduplicate Data
expanded_results = []
for entry in results:
    pos_a, pos_b = entry['poses_compared']
    
    # Instance for POS A
    row_a = entry.copy()
    row_a['focus_pos'] = pos_a
    row_a['comp_pos'] = pos_b
    expanded_results.append(row_a)
    
    # Instance for POS B
    row_b = entry.copy()
    row_b['focus_pos'] = pos_b
    row_b['comp_pos'] = pos_a
    expanded_results.append(row_b)

df_expanded = pd.DataFrame(expanded_results)

# Remove exact duplicates if they exist in your source list
df_expanded = df_expanded.drop_duplicates(subset=['lemma', 'focus_pos', 'comp_pos', 'split_quality_score'])

# 2. Generate the Plot
fig = px.box(
    df_expanded, 
    x="comp_pos",
    y="split_quality_score", 
    facet_col="focus_pos",
    facet_col_wrap=3,
    color="comp_pos",
    points="all",           # This shows the points
    hover_name="lemma",
    title="Category Health: Points Cleaned & De-duplicated",
)

# 3. Adjust "Jitter" and Point Positioning
# This spreads the points out so they don't overlap vertically/horizontally
fig.update_traces(pointpos=0, jitter=0.3) 

# 4. Axes and Layout Fixes
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
sq_thresh = df['split_quality_score'].median()
fig.add_hline(y=sq_thresh, line_dash="dot", line_color="red", annotation_text="Median")
fig.update_layout(
    height=1700,
    # template="plotly_white",
    # hoverlabel=dict(bgcolor="white", font_size=12)
)
fig.show()